# **(2)** Pre-Training Chemprop (multi-task on proxy properties)

Here we **pretrain** chemprop on four physicochemical properties of peptides.  
The resulting model will used for **transfer learning** on a smaller dataset of experimental targets (purity, solubility, hemolysis).


### Why pretraining?

| | Direct Training | Pre-Training |
|---|---|---|
| **Data needed** | Many labelled samples | Few labelled samples |
| **Backbone** | Randomly initialised | Already learned molecular representations |
| **Transfer** | X | frozen layers |

### Pre-Training targets (proxy properties, only for canonical aa)

| # | Column | Description |
|---|--------|-------------|
| 1 | `gravy` | Grand Average of Hydropathy |
| 2 | `instability` | proxy of half-life |
| 3 | `charge` | net charge at pH=7.4 |
| 4 | `diff_coupling` | coupling score for difficulty in peptide synthesis |


In [ ]:
pip install chemprop lightning rdkit pandas matplotlib

In [ ]:
import os

if os.getenv("COLAB_RELEASE_TAG"):
    if not os.path.exists("IX_technical_turorial2026"):
        !git clone https://github.com/rbirolo/IX_technical_turorial2026.git
    
    %cd IX_technical_turorial2026


In [ ]:
import warnings, pickle
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lightning as pl
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

from chemprop.data import (
    MoleculeDatapoint, MoleculeDataset, build_dataloader,
    make_split_indices, split_data_by_indices,
)
from chemprop.models import MPNN
from chemprop.nn import BondMessagePassing, MeanAggregation, RegressionFFN, RMSE, MAE

## Configuration

Hyperparameters and set up.


In [ ]:
# input file, SMILES: 4 targets
data_file  = "data/pretraining_set_15k.csv"
MT_output  = Path("chemprop_pretraining")       # folder where weights are saved
targets    = ["gravy", "instability", "charge", "diff_coupling"]

# Architecture
MP_DEPTH   = 3      # message-passing depth (number of bond iterations)
MP_DIM     = 300    # hidden dimension of the MP network
FFN_LAYERS = 2      # number of hidden layers in the feed-forward head
FFN_DIM    = 300    # hidden dimension of the FFN
DROPOUT    = 0.2

# Training
MAX_EPOCHS = 50
INIT_LR    = 1e-4
MAX_LR     = 1e-3
FINAL_LR   = 1e-4
WARMUP_EPS = 2
BATCH_SIZE = 64
SEED       = 42

SPLIT_SIZES = (0.70, 0.15, 0.15)   # train / val / test

## Data Loading & Cleaning

The CSV contains a `SMILES` column plus one column per target.  
As in the previous notebook, for preparing the data:
1. Parse each SMILES with RDKit.
2. Coerce targets to `float`.  
3. Print a summary table to sanity-check the dataset.


In [ ]:
def load_dataset(data_file: str):
    df = pd.read_csv(data_file)

    for t in targets:
        df[t] = pd.to_numeric(df[t], errors="coerce")
    df = df.dropna(subset=["SMILES"]).copy()

    mols, valid_idx = [], []
    for i, smi in enumerate(df["SMILES"]):
        mol = Chem.MolFromSmiles(str(smi))
        if mol is not None:
            mols.append(mol)
            valid_idx.append(i)
        else:
            print(f"Invalid SMILES (row {i}): {str(smi)[:80]}…")

    df  = df.iloc[valid_idx].reset_index(drop=True)
    ys  = df[targets].values.astype(float)
    return mols, ys, df

mols, ys, df_clean = load_dataset(data_file)
N = len(mols)

print(f"{'Target':<22} {'entries':>8} {'mean':>9} {'std':>7} {'min':>8} {'max':>8}")
print("─" * 66)
for i, t in enumerate(targets):
    v = ys[:, i][~np.isnan(ys[:, i])]
    print(f"{t:<22} {len(v):>8} {v.mean():>9.4f} {v.std():>7.4f} {v.min():>8.4f} {v.max():>8.4f}")

## Targets Distribution


In [ ]:
fig, axes = plt.subplots(1, len(targets), figsize=(5.5 * len(targets), 5))
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B3"]

for ax, t, c in zip(axes, targets, colors):
    v = ys[:, targets.index(t)]
    v = v[~np.isnan(v)]
    ax.hist(v, bins=30, color=c, edgecolor="white", linewidth=0.6)
    ax.axvline(v.mean(), color="black", ls="--", lw=1.2, label=f"μ={v.mean():.2f}")
    ax.set_title(t, fontsize=12, fontweight="normal")
    ax.set_xlabel("value")
    ax.set_ylabel("count")
    ax.legend(fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Pre-training target distributions", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Train / Val / Test Split


In [ ]:
pl.seed_everything(SEED)

train_idxs, val_idxs, test_idxs = make_split_indices(
    mols, split="random", sizes=SPLIT_SIZES, seed=SEED
)

_train_idx = train_idxs[0]
_val_idx   = val_idxs[0]
_test_idx  = test_idxs[0]

print(f"Train : {len(_train_idx):>5}  ({len(_train_idx)/N*100:.1f}%)")
print(f"Val   : {len(_val_idx):>5}  ({len(_val_idx)/N*100:.1f}%)")
print(f"Test  : {len(_test_idx):>5}  ({len(_test_idx)/N*100:.1f}%)")

## Functions

Define three functions for (i) preparing the input; (ii) build the model; and (iii) define the training:

| Function | What it does |
|----------|--------------|
| `make_loaders` | Splits datapoints, normalises targets on train, returns DataLoaders |
| `make_mpnn` | Builds a `BondMessagePassing → MeanAggregation → RegressionFFN` model |
| `run_trainer` | Trains with Lightning's `Trainer` |


In [ ]:
def make_loaders(datapoints):
    # split_data_by_indices to split in train, val, test
    # trasform the data in a dataset of graphs with MoleculeDataset
    # nornalise the labels
    # dataloader

    return train_load, val_load, test_load, scaler


def make_mpnn(n_tasks: int):
    # create the model architecture
    # step1 message passing
    # step2 aggregation
    # step3 FFN

    return MPNN(mp, agg, ffn, metrics=[RMSE(), MAE()])

def run_trainer(model, train_load, val_load):
    # define and fit the trainer

    return trainer



In [ ]:
# @title Solutions
def make_loaders(datapoints):
    train, val, test = split_data_by_indices(datapoints, train_idxs, val_idxs, test_idxs)
    train = [dp for sub in train for dp in sub]
    val   = [dp for sub in val   for dp in sub]
    test  = [dp for sub in test  for dp in sub]

    train_ds = MoleculeDataset(train)
    val_ds   = MoleculeDataset(val)
    test_ds  = MoleculeDataset(test)

    scaler = train_ds.normalize_targets()
    val_ds.normalize_targets(scaler)

    train_load = build_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_load   = build_dataloader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_load  = build_dataloader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_load, val_load, test_load, scaler


def make_mpnn(n_tasks: int) -> MPNN:
    mp  = BondMessagePassing(depth=MP_DEPTH, d_h=MP_DIM, dropout=DROPOUT)
    agg = MeanAggregation()
    ffn = RegressionFFN(
        n_tasks   = n_tasks,
        input_dim = mp.output_dim,
        hidden_dim= FFN_DIM,
        n_layers  = FFN_LAYERS,
        dropout   = DROPOUT,
    )
    return MPNN(mp, agg, ffn, metrics=[RMSE(), MAE()])


def run_trainer(model, train_load, val_load):
    trainer = pl.Trainer(
        max_epochs = MAX_EPOCHS,
        enable_progress_bar = True,
        enable_model_summary = False,
        logger = False,
    )
    trainer.fit(model, train_load, val_load)
    return trainer


## Build Multi-Task Datapoints

Each datapoint carries `y` of shape `(4,)`, i.e. one value per target


In [ ]:
mt_pretrain_datapoints = [
    MoleculeDatapoint(mol=m, y=y.flatten())
    for m, y in zip(mols, ys)
]

print(f"{len(mt_pretrain_datapoints)} datapoints  |  y shape: ({len(targets)},)")

## Training

We build the loaders, instantiate a **4-head MPNN**, and train it.

```
SMILES → BondMessagePassing (shared backbone)
       → MeanAggregation
       → RegressionFFN (n_tasks=4)
       → [gravy, instability, charge, diff_coupling]
```

> ⏱ With `MAX_EPOCHS=50` and 15k entries this takes ~15 min on GPU.  
> Reduce `MAX_EPOCHS` for a quick test.
 Directly upload the pretrained model for model evaluation (load checkpoints).


In [ ]:
#load the data

#define the model Multi-task with 4 heads

#start the training



In [ ]:
# @title Solution
mt_pre_train_load, mt_pre_val_load, mt_pre_test_load, mt_pre_scaler = make_loaders(mt_pretrain_datapoints)

mt_model  = make_mpnn(n_tasks=len(targets))
n_params  = sum(p.numel() for p in mt_model.parameters())
print(f"Parameters: {n_params:,}  |  n_tasks={len(targets)}")

run_trainer(mt_model, mt_pre_train_load, mt_pre_val_load)

## Save the model

- **model weights** saved in `chemprop_pretraining/pretraining_multitask_model.pt`
- **target scaler** saved in `chemprop_pretraining/pretraining_multitask_scaler.pkl`

The scaler is needed to un-normalise predictions at inference time.

❗**For time reasons, skip the training and model saving and upload the pre-saved model to evaluate the model performance.**

In [ ]:
MT_output.mkdir(parents=True, exist_ok=True)

torch.save(mt_model.state_dict(), MT_output / "pretraining_multitask_model.pt")
with open(MT_output / "pretraining_multitask_scaler.pkl", "wb") as f:
    pickle.dump(mt_pre_scaler, f)

## Inspect the checkpoint

Verify the saved weights and the layers layout.  
This helps in knowing which keys to load and which to re-initialise for the next step of transfer-learning.

| Key pattern | Role |
|---|---|
| `message_passing.*` | Shared backbone (will be **frozen** in FT) |
| `predictor.ffn.0.0.*` | Hidden layer 1 (will be **trainable** in FT) |
| `predictor.ffn.1.2.*` | Hidden layer 2 (will be **trainable** in FT) |
| `predictor.ffn.2.2.*` | Output layer — shape `[4, 300]` (will be **re-initialised** for 3 heads in FT) |


In [ ]:
ckpt = torch.load("models/chemprop_pretraining/pretraining_multitask_model.pt", weights_only=True)

print(f"Total keys in checkpoint: {len(ckpt)}\n")
for key, tensor in ckpt.items():
    print(f"  {key:<55} {str(tuple(tensor.shape))}")

## Evaluate on the test set

1. Run the model on the test loader.
2. Un-normalise predictions with the saved scaler.
3. Computes RMSE, MAE, Pearson r, Spearman r per target.


In [ ]:
from scipy import stats as sp_stats
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate_mt(model, test_loader, scaler, target_names):
    model.eval()
    all_preds, all_gts = [], []

    with torch.no_grad():
        for batch in test_loader:
            preds = model(batch.bmg, batch.V_d, batch.X_d)
            all_preds.append(preds.cpu().numpy())
            all_gts.append(batch.Y.cpu().numpy())

    preds = np.vstack(all_preds)
    gts   = np.vstack(all_gts)

    # un-normalise
    preds = scaler.inverse_transform(preds)

    metrics = {}
    for i, t in enumerate(target_names):
        gt_i = gts[:, i]; pr_i = preds[:, i]
        mask = ~np.isnan(gt_i)
        gt_i, pr_i = gt_i[mask], pr_i[mask]
        if len(gt_i) < 2:
            continue
        metrics[t] = {
            "rmse"      : float(np.sqrt(mean_squared_error(gt_i, pr_i))),
            "mae"       : float(mean_absolute_error(gt_i, pr_i)),
            "pearson_r" : float(sp_stats.pearsonr(gt_i, pr_i)[0]),
            "spearman_r": float(sp_stats.spearmanr(gt_i, pr_i)[0]),
            "n"         : int(mask.sum()),
            "gt"        : gt_i,
            "pr"        : pr_i,
        }
    return metrics

mt_pre_metrics = evaluate_mt(mt_model, mt_pre_test_load, mt_pre_scaler, targets)

print(f"\n{'Target':<22} {'RMSE':>8} {'MAE':>8} {'Pearson':>9} {'Spearman':>10} {'Number datapoints':>6}")
print("─" * 76)
for t in targets:
    if t in mt_pre_metrics:
        m = mt_pre_metrics[t]
        print(f"  {t:<20}  {m['rmse']:>8.4f} {m['mae']:>8.4f} {m['pearson_r']:>9.4f} {m['spearman_r']:>10.4f} {m['n']:>6}")

## Predicted vs original labels plot

In [ ]:
fig, axes = plt.subplots(1, len(targets), figsize=(5.5 * len(targets), 5))
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B3"]

for ax, t, c in zip(axes, targets, colors):
    if t not in mt_pre_metrics:
        ax.set_visible(False)
        continue
    m  = mt_pre_metrics[t]
    lo = min(m["gt"].min(), m["pr"].min()) * 0.95
    hi = max(m["gt"].max(), m["pr"].max()) * 1.05

    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x", zorder=1)
    ax.scatter(m["gt"], m["pr"], color=c, edgecolors="white", s=55,
               linewidths=0.5, alpha=0.85, zorder=2,
               label=f"Pearson r={m['pearson_r']:.3f}")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_title(t, fontsize=12, fontweight="normal")
    ax.set_xlabel("y_value"); ax.set_ylabel("predicted")
    ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()

## External validation, prediction on new peptides outside the dataset

Run predictions on a new list of peptides (SMILES on external_150_pretraining.csv).
To check model generalisation and prediction befor fine-tuning.


In [ ]:
def predict(smiles_list: list):
    with open("models/chemprop_pretraining/pretraining_multitask_scaler.pkl", "rb") as f:    #load the model
        sc = pickle.load(f)

    ckpt_inf = torch.load("models/chemprop_pretraining/pretraining_multitask_model.pt", weights_only=True)
    n_tasks  = ckpt_inf["predictor.ffn.2.2.bias"].shape[0]
    model_inf = make_mpnn(n_tasks=n_tasks)
    model_inf.load_state_dict(ckpt_inf)
    model_inf.eval()

    mols_new, valid_smiles = [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is not None:
            mols_new.append(mol); valid_smiles.append(smi)
        else:
            print(f"Invalid SMILES (skipped): {smi}")

    dps    = [MoleculeDatapoint(mol=m) for m in mols_new]
    ds     = MoleculeDataset(dps)
    loader = build_dataloader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    preds = []
    with torch.no_grad():
        for batch in loader:
            p = model_inf(batch.bmg, batch.V_d, batch.X_d)
            preds.append(p.cpu().numpy())

    preds = sc.inverse_transform(np.vstack(preds))
    return pd.DataFrame(preds, columns=targets, index=valid_smiles)

In [ ]:
#Load data from csv file
df_ext     = pd.read_csv("data/150_external_proxy.csv")
new_smiles = df_ext["SMILES"].tolist()

df_pred = predict(new_smiles)
print(f"Predictions for {len(df_pred)} molecules:")
df_pred.round(4)

## 13 · External Set: Predicted vs Labelled


In [ ]:
fig, axes = plt.subplots(1, len(targets), figsize=(5.5 * len(targets), 5))
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B3"]

for ax, t, c in zip(axes, targets, colors):
    if t not in df_ext.columns:
        ax.text(0.5, 0.5, f"'{t}'\nnot in external CSV",
                ha="center", va="center", transform=ax.transAxes, fontsize=10)
        ax.set_visible(False)
        continue

    gt   = df_ext[t].values
    pr   = df_pred[t].values
    mask = ~np.isnan(gt)
    gt, pr = gt[mask], pr[mask]

    lo = min(gt.min(), pr.min()) * 0.95
    hi = max(gt.max(), pr.max()) * 1.05

    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x", zorder=1)
    ax.scatter(gt, pr, color=c, edgecolors="white", s=65,
               linewidths=0.5, alpha=0.85, zorder=2)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_title(t, fontsize=12, fontweight="normal")
    ax.set_xlabel("y_value"); ax.set_ylabel("predicted")
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()